---
# **PROGETTO "SPATIAL STAY-POINT DETECTION" (k-means, DBSCAN) - BIANCHI GALLOTTA**
---

# ▶️ CUDA setup

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


In [2]:
!nvidia-smi

Mon Nov  6 09:17:46 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   48C    P8     9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [3]:
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb
!apt update
!apt install ./nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb
!apt --fix-broken install

--2023-11-06 09:17:46--  https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 152.195.19.142
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|152.195.19.142|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 317705436 (303M) [application/x-deb]
Saving to: ‘nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb’

nsight-systems-2023 100%[===================>] 302.99M   269MB/s    in 1.1s    

2023-11-06 09:17:48 (269 MB/s) - ‘nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb’ saved [317705436/317705436]

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InR

## NVCC Plugin for Jupyter notebook

*Usage*:


*   Load Extension `%load_ext nvcc_plugin`
*   Mark a cell to be treated as cuda cell
`%%cuda --name example.cu --compile false`

**NOTE**: The cell must contain either code or comments to be run successfully. It accepts 2 arguments. `-n | --name` - which is the name of either CUDA source or Header. The name parameter must have extension `.cu` or `.h`. Second argument -c | --compile; default value is false. The argument is a flag to specify if the cell will be compiled and run right away or not. It might be usefull if you're playing in the main function

*  We are ready to run CUDA C/C++ code right in your Notebook. For this we need explicitly say to the interpreter, that we want to use the extension by adding `%%cu` at the beginning of each cell with CUDA code.




In [4]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-g1_fh_7u
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-g1_fh_7u
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 0a71d56e5dce3ff1f0dd2c47c29367629262f527
  Preparing metadata (setup.py) ... done
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-py3-none-any.whl size=4295 sha256=ef3d57a050d1e3d9cd053a45488c9c83cb40f51077caffdd1b2462d734bf24dd
  Stored in directory: /tmp/pip-ephem-wheel-cache-7bs48i1d/wheels/a8/b9/18/23f8ef71ceb0f63297dd1903aedd067e6243a68ea756d6feea
Successfully built NVCCPlugin


In [5]:
%load_ext nvcc_plugin

created output directory at /content/src
Out bin /content/result.out


In [6]:
# plugin for cpp sintax highlighting

!wget -O cpp_plugin.py https://gist.github.com/akshaykhadse/7acc91dd41f52944c6150754e5530c4b/raw/cpp_plugin.py
%load_ext cpp_plugin

--2023-11-06 09:18:22--  https://gist.github.com/akshaykhadse/7acc91dd41f52944c6150754e5530c4b/raw/cpp_plugin.py
Resolving gist.github.com (gist.github.com)... 140.82.112.3
Connecting to gist.github.com (gist.github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/akshaykhadse/7acc91dd41f52944c6150754e5530c4b/raw/cpp_plugin.py [following]
--2023-11-06 09:18:22--  https://gist.githubusercontent.com/akshaykhadse/7acc91dd41f52944c6150754e5530c4b/raw/cpp_plugin.py
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2730 (2.7K) [text/plain]
Saving to: ‘cpp_plugin.py’

cpp_plugin.py       100%[===================>]   2.67K  --.-KB/s    in 0s      

2023-11-06 09:18:23 (4

# ▶️ Device Query

In [7]:
# DeviceQuery dell'attuale device (su Colab!)
!nvcc /content/GPUcomputing/utils/deviceQuery.cu -o deviceQuery
!./deviceQuery

cc1plus: fatal error: /content/GPUcomputing/utils/deviceQuery.cu: No such file or directory
compilation terminated.
/bin/bash: line 1: ./deviceQuery: No such file or directory


Check whether the device can transfer in both directions simultaneously

In [8]:
%%cu
#include <stdio.h>

int main(void) {
    cudaDeviceProp dProp;
    cudaGetDeviceProperties(&dProp, 0);

    // Shows whether the device can transfer in both directions simultaneously
    printf("Device %s capable of simultaneous CPU-to-GPU and GPU-to-CPU datatransfers\n", dProp.deviceOverlap ? "IS": "NOT");

    return 0;
}

Device IS capable of simultaneous CPU-to-GPU and GPU-to-CPU datatransfers



# ▶️ Data Management and Common Operations

In [9]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [10]:
%%cuda --name common.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <float.h>

#define N_SAMPLES 32372
#define USER_LENGTH 17

// # Struttura di array: utenti, array coordinate x, array coordinate y
struct Samples {
    char user[N_SAMPLES][USER_LENGTH];
    float x[N_SAMPLES];
    float y[N_SAMPLES];
};

// # Rimuove le virgolette da un token.
char* remove_quotes(const char *token) {
    int len = strlen(token);
    char *result = (char*)malloc(len + 1);

    if (result == NULL) {
        perror("[remove_quotes] errore nell'allocazione di memoria");
        exit(1);
    }

    int i, j = 0;
    for (i = 0; i < len; i++) {
        if (token[i] != '"') {
            result[j++] = token[i];
        }
    }

    result[j] = '\0';

    return result;
}

// # Legge il csv con i dati e gli inserisce nella struttura "Samples".
int read_samples(struct Samples *dataset) {
    FILE *file = fopen("/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/dataset/example_trajectory_students.csv", "r");

    if (file == NULL) {
        perror("[read_samples] impossibile aprire il file specificato");
        return 1;
    }

    int row = 0;
    char line[1024];

    if (fgets(line, sizeof(line), file) == NULL) {
        perror("[read_samples] impossibile leggere dal file");
        fclose(file);
        return 1;
    }

    while (fgets(line, sizeof(line), file)) {
        char *token = strtok(line, ",");

        strcpy(dataset->user[row], remove_quotes(token));
        strtok(NULL, ","); // # Si salta il timestamp.
        dataset->x[row] = atof(remove_quotes(strtok(NULL, ",")));
        dataset->y[row] = atof(remove_quotes(strtok(NULL, ",")));

        row++;
    }

    fclose(file);

    return 0;
}

// # Scrive i samples accompagnati dall'id del cluster di appartenenza in un file.
int write_samples(struct Samples *dataset, int *cluster, bool is_pkmeans) {
    FILE *file = fopen(is_pkmeans ? "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/dataset/pkmeans_trajectory_students.csv" : "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/dataset/gdbscan_trajectory_students.csv", "w");

    if (file == NULL) {
        perror("[write_samples] impossibile aprire il file specificato");
        return 1;
    }

    fprintf(file, "person_id,x,y,cluster\n");

    for (int i = 0; i < N_SAMPLES - 1; i++) {
        fprintf(file, "%s,%f,%f,%d\n", dataset->user[i], dataset->x[i], dataset->y[i], cluster[i]);
    }

    fclose(file);

    return 0;
}

// # Calcola la distanza euclidea tra due punti.
float calculate_distance(float x1, float y1, float x2, float y2) {
    return sqrt(pow(x1 - x2, 2) + pow((y1 - y2), 2));
}

// # Calcola la distanza euclidea tra due punti su device.
__device__ float d_calculate_distance(float x1, float y1, float x2, float y2) {
    return sqrt(pow(x1 - x2, 2) + pow((y1 - y2), 2));
}

// # Controlla che i due array passati come parametro siano uguali.
int arrays_match(int *arr1, int *arr2) {
    for (int i = 0; i < N_SAMPLES; i++) {
        if (arr1[i] != arr2[i]) {
            printf("Atteso valore CPU \"%d\" in posizione %d, ottenuto in GPU \"%d\"\n", arr1[i], i, arr2[i]);
            return 0;
        }
    }

    return 1;
}

'File written in /content/src/common.cu'

# ✅ Parallel k-means with MapReduce

In [11]:
%%cuda --name map.cu

#include <stdio.h>
#include <string.h>

#define MAX_ENTRIES 32372

// # Struttura della mappa: array delle chiavi (key), array delle coordinate x, array delle coordinate y, dimensione variabile della mappa (size).
struct KeyValueMap {
    int key[MAX_ENTRIES];
    float x[MAX_ENTRIES];
    float y[MAX_ENTRIES];
    int size;
};

// # Inizializzazione della mappa.
void init_map(struct KeyValueMap *map) {
    map->size = 0;
    memset(&map->key, 0, sizeof(&map->key));
    memset(&map->x, 0, sizeof(&map->x));
    memset(&map->y, 0, sizeof(&map->y));
}

//# Funzione 'add' CPU.
void add_to_map(struct KeyValueMap *map, int new_key, float new_x, float new_y) {
    if (map->size < MAX_ENTRIES) {
        map->key[map->size] = new_key;
        map->x[map->size] = new_x;
        map->y[map->size] = new_y;
        map->size++;
    } else {
        printf("[add_to_map] la mappa è piena (%d). Impossibile aggiungere ulteriori elementi.\n", map->size);
    }
}

// # Funzione device 'add' GPU (eseguita da un thread).
__device__ bool d_add_to_map(struct KeyValueMap *map, int index, int new_key, float new_x, float new_y) {
    if (index < MAX_ENTRIES) {
        map->key[index] = new_key;
        map->x[index] = new_x;
        map->y[index] = new_y;
    } else {
        printf("[add_to_map] la mappa è piena (%d). Impossibile aggiungere ulteriori elementi.\n", map->size);
    }
}

// # Funzione 'search_count' CPU: CONTA il numero di elementi associati alla stessa chiave.
int search_count_in_map(struct KeyValueMap *map, int key) {
    int cont = 0;

    for (int i = 0; i < map->size; i++) {
        if (map->key[i] == key) {
            cont ++;
        }
    }

    return cont;
}

// # Funzione device 'search_count' GPU (eseguita da un thread).
__device__ int d_search_count_in_map(struct KeyValueMap *map, int key) {
    int cont = 0;

    for (int i = 0; i < map->size; i++) {
        if (map->key[i] == key) {
            cont ++;
        }
    }

    return cont;
}

// # Funzione 'search' CPU: TROVA il numero di elementi associati alla stessa chiave e se non ce ne sono torna 'falso' altrimenti 'vero'.
// # Negli array found_xs e found_ys si trovano le coordinate x e y associate alla chiave 'key'.
bool search_in_map(struct KeyValueMap *map, int key, float *found_xs, float *found_ys) {
    int cont = 0;

    for (int i = 0; i < map->size; i++) {
        if (map->key[i] == key) {
            found_xs[cont] = map->x[i];
            found_ys[cont] = map->y[i];
            cont ++;
        }
    }

    return cont != 0;
}

// # Funzione device 'search' GPU (eseguita da un thread).
__device__ bool d_search_in_map(struct KeyValueMap *map, int key, float *found_xs, float *found_ys) {
    int cont = 0;

    for (int i = 0; i < map->size; i++) {
        if (map->key[i] == key) {
            found_xs[cont] = map->x[i];
            found_ys[cont] = map->y[i];
            cont ++;
        }
    }

    return cont != 0;
}

'File written in /content/src/map.cu'

In [12]:
%%cuda --name pkmeans.cu

#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/common.h"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph.h"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph.cpp"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph_d.cu"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph_d.h"
#include "/content/src/common.cu"
#include "/content/src/map.cu"
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <float.h>

// # Parametro scelto
#define BLOCK_SIZE 1024

#define K 23
#define E 1e-10

// # Struttura contenente le coordinate dei 23 centri (x,y)
struct Centers {
    float x[K];
    float y[K];
};

// # Struttura contenente il tempo di calcolo di un algoritmo.
struct TimedResult {
    double total;
    int* cluster;
};

// #>>>>>>>>>> FUNZIONI COMUNI DI INIZIALIZZAZIONE <<<<<<<<<<

// # Inizializzazione dei K = 23 centri (k-means++).
int init_centers(struct Centers *centers, struct Samples *samples) {
    srand(time(NULL));

    // # Si sceglie il primo centro casuale tra i campioni.
    int first_center_idx = rand() % N_SAMPLES;
    centers->x[0] = samples->x[first_center_idx];
    centers->y[0] = samples->y[first_center_idx];

    // # Inizializzazione di un array per contenere le distanze minime tra i campioni e i centri.
    float *min_distances = (float *)malloc(N_SAMPLES * sizeof(float));

    for (int i = 1; i < K; i++) {
        float total_distance = 0.0;

        // # Calcolo delle distanze minime tra i campioni e i centri già scelti.
        for (int j = 0; j < N_SAMPLES; j++) {
            float min_distance = calculate_distance(samples->x[j], samples->y[j], centers->x[0], centers->y[0]);

            for (int k = 1; k < i; k++) {
                float distance = calculate_distance(samples->x[j], samples->y[j], centers->x[k], centers->y[k]);
                if (distance < min_distance) {
                    min_distance = distance;
                }
            }

            min_distances[j] = min_distance;
            total_distance += min_distance;
        }

        // # Scelta di un nuovo centro pesato dalle distanze minime.
        float random_value = ((float)rand() / RAND_MAX) * total_distance;
        int new_center_idx = -1;

        for (int j = 0; j < N_SAMPLES; j++) {
            random_value -= min_distances[j];
            if (random_value <= 0) {
                new_center_idx = j;
                break;
            }
        }

        centers->x[i] = samples->x[new_center_idx];
        centers->y[i] = samples->y[new_center_idx];
    }

    free(min_distances);

    return 0;
}

// # Copia dei K = 23 centri.
int copy_centers(struct Centers *dst, struct Centers *src) {
    for (int i = 0; i < K; i ++) {
        dst->x[i] = src->x[i];
        dst->y[i] = src->y[i];
    }

    return 0;
}

// # Creazione della struttura che misura i tempi di esecuzione.
TimedResult make_timed_result(int *cluster) {
    TimedResult result;
    result.cluster = cluster;

    return result;
}

// #>>>>>>>>>> FUNZIONI ALGORITMO CPU <<<<<<<<<<

// # Assegno ad ogni punto del dataset (indicato con key) ad un cluster in base al centro più vicino.
int assign_cluster(struct Samples *dataset, int key, struct Centers *centers, struct KeyValueMap *map, int *clusters) {
    double min_dis = DBL_MAX;
    int index = -1;

    // # Calcolo della distanza tra il punto e tutti i centri per trovare quello più vicino.
    for (int i = 0; i < K; i++) {
        double dis = calculate_distance(dataset->x[key], dataset->y[key], centers->x[i], centers->y[i]);

        if (dis < min_dis) {
            min_dis = dis;
            index = i;
        }
    }

    clusters[key] = index;  // # Associo il cluster al punto del dataset.
    add_to_map(map, index, dataset->x[key], dataset->y[key]); // # (struct KeyValueMap *map, int new_key, float new_x, float new_y) -> Al centro associo le x e le y.

    return 0;
}

// # Ciclo su ogni campione del cluster e gli associo un centro.
int explore_samples(struct Samples *dataset, struct Centers *centers, struct KeyValueMap *map, int *clusters) {
    for (int i = 0; i < N_SAMPLES; i++) {
        assign_cluster(dataset, i, centers, map, clusters);
    }

    return 0;
}

// # Cicla sui punti del cluster e ricarica il nuovo centro (x,y) facendo la media.
int calculate_new_centroids(int key, int nums, float *samples_x, float *samples_y, struct Centers *centers) {
    float sum_x = 0;
    float sum_y = 0;

    for (int i = 0; i < nums; i ++) {
        sum_x += samples_x[i];
        sum_y += samples_y[i];
    }

    centers->x[key] = sum_x / nums;
    centers->y[key] = sum_y / nums;

    return 0;
}

// # Funzione usata per aggiornare i centroidi.
int update_centroids(struct KeyValueMap *map, struct Centers *centers) {
    // # Conto il numero di punti per cluster (centroide).
    int somma = 0;

    for (int i = 0; i < K; i++) {
        int nums = search_count_in_map(map, i);

        if (nums == 0) {  // # Se il cluster non ha punti continuo.
            continue;
        }

        float samples_x[nums], samples_y[nums];

        if (!search_in_map(map, i, samples_x, samples_y)) { // # Se il cluster non ha punti continuo, altrimenti salvo i punti dentro samples_x e samples_y.
            perror("[update_centroids] nessun valore trovato. (per search_in_map)");
            continue;
        }

        // # Calcolo il nuovo centroide dando in input: la chiave i, il numero dei punti associati al cluster con chiave i, la lista dei punti x, la lista dei punti y e la lista dei centri da aggiornare.
        calculate_new_centroids(i, nums, samples_x, samples_y, centers);
        somma += nums;
    }

    return 0;
}

// # Controlla che la distanza tra il centro precedente e il nuovo centro sia maggiore di una soglia E (se minore i 2 centri sono considerati uguali).
bool check_threshold(struct Centers *centers, struct Centers *new_centers) {
    for (int i = 0; i < K; i ++) {
        if (calculate_distance(centers->x[i], centers->y[i], new_centers->x[i], new_centers->y[i]) >= E) {
            return false;
        }
    }

    return true;
}

// # Stampa un messaggio di warning se ci sono dei cluster vuoti.
int warn_for_empty_centers(struct KeyValueMap *map) {
    int i = 0;

    for (i = 0; i < K; i ++) {
        if (search_count_in_map(map, i) == 0) {
            break;
        }
    }

    if (i < K) printf("⚠ Ci sono dei cluter vuoti\n");

    return 0;
}

// # Algoritmo pk-means su CPU.
TimedResult cpu_pkmeans(struct Samples *dataset, struct Centers *centers) {
    // # Stampa dei centri iniziali.
    printf("PKMEANS CPU:\n  - Definizione delle strutture dati...\n");

    // # Definizione della mappa usata dall'algoritmo.
    struct KeyValueMap *map;
    map = (struct KeyValueMap*) malloc(sizeof(struct KeyValueMap));
    printf("  - Definizione completata.\n  - Inizializzazione in corso...\n");

    // # Popolamento del dataset.
    read_samples(dataset);

    printf("  - Inizializzazione completata.\n  - Inizio dell'algoritmo...\n");

    bool stop_condition = false;
    int cont = 0;
    int clusters[N_SAMPLES];

    while (!stop_condition) {
        // # Reset della mappa ad ogni iterazione.
        init_map(map);

        // # Assegno ad ogni campione del cluster un centro.
        explore_samples(dataset, centers, map, clusters);

        // # Creo una struttura dei nuovi centri.
        struct Centers *new_centers;
        new_centers = (struct Centers*) malloc(sizeof(struct Centers));
        copy_centers(new_centers, centers);

        // # Aggiorno i centroidi salvandoli in new_centers.
        update_centroids(map, new_centers);

        // # Controllo che i vecchi centri e i nuovi abbiano una distanza minima.
        stop_condition = check_threshold(centers, new_centers);

        // # Salvo le evoluzioni dei centri in un array.
        centers = new_centers;

        cont ++;
    }

    printf("  - Algoritmo terminato in %d iterazioni.\n✔ PKMEANS CPU completato\n", cont);

    warn_for_empty_centers(map);

    free(map);
    free(centers);

    // # Cluster è un array di N_SAMPLES contenente per ogni campione l'indice del cluster a cui è associato.
    return make_timed_result(clusters);
}

// #>>>>>>>>>> FUNZIONI ALGORITMO GPU <<<<<<<<<<

// # Assegno ad ogni punto del dataset (indicato con key) ad un cluster in base al centro più vicino su GPU.
__global__ void d_assign_cluster(struct Samples *dataset, struct Centers *centers, struct KeyValueMap *map, int *clusters) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx >= N_SAMPLES) return;

    double min_dis = DBL_MAX;
    int index = -1;

    for (int i = 0; i < K; i++) {
        double dis = d_calculate_distance(dataset->x[idx], dataset->y[idx], centers->x[i], centers->y[i]);

        if (dis < min_dis) {
            min_dis = dis;
            index = i;
        }
    }

    clusters[idx] = index;

    d_add_to_map(map, idx, index, dataset->x[idx], dataset->y[idx]);
}

// # Cicla sui punti del cluster e ricarica il nuovo centro (x,y) facendo la media su GPU.
__device__ void d_calculate_new_centroids(int key, int nums, float *samples_x, float *samples_y, struct Centers *centers) {
    float sum_x = 0;
    float sum_y = 0;

    for (int i = 0; i < nums; i ++) {
        sum_x += samples_x[i];
        sum_y += samples_y[i];
    }

    centers->x[key] = sum_x / nums;
    centers->y[key] = sum_y / nums;
}

// # Funzione usata per aggiornare i centroidi su GPU.
__global__ void d_update_centroids(struct KeyValueMap *map, struct Centers *centers) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx > K) return;

    int nums = d_search_count_in_map(map, idx);

    float samples_x[N_SAMPLES];
    float samples_y[N_SAMPLES];

    if (nums == 0) {
        return;
    }

    if (!d_search_in_map(map, idx, samples_x, samples_y)) {
        printf("[update_centroids] nessun valore trovato. (per search_in_map)\n");
        return;
    }

    d_calculate_new_centroids(idx, nums, samples_x, samples_y, centers);
}

// # Algoritmo pk-means su GPU.
TimedResult gpu_pkmeans(struct Samples *h_dataset, struct Centers *h_centers) {
    printf("PKMEANS GPU:\n  - Definizione delle strutture dati...\n");

    // # Definizione e allocazione della mappa su CPU e GPU.
    struct KeyValueMap *h_map, *d_map;
    h_map = (struct KeyValueMap*) malloc(sizeof(struct KeyValueMap));
    CHECK(cudaMalloc((void **) &d_map, sizeof(struct KeyValueMap)));

    // # Definizione dei centri su CPU e GPU.
    struct Centers *d_centers;
    CHECK(cudaMalloc((void **) &d_centers, sizeof(struct Centers)));
    CHECK(cudaMemcpy(d_centers, h_centers, sizeof(struct Centers), cudaMemcpyHostToDevice));

    printf("  - Definizione completata.\n  - Inizializzazione in corso...\n");

    // # Lettura dei campioni dal dataset e allocazione su GPU.
    read_samples(h_dataset);
    struct Samples *d_dataset;
    CHECK(cudaMalloc((void **) &d_dataset, sizeof(struct Samples)));
    CHECK(cudaMemcpy(d_dataset, h_dataset, sizeof(struct Samples), cudaMemcpyHostToDevice));

    printf("  - Inizializzazione completata.\n  - Inizio dell'algoritmo...\n");

    bool stop_condition = false;
    int cont = 0;

    // # Allocazione dell'array dei cluster su CPU e GPU.
    int h_clusters[N_SAMPLES];
    int *d_clusters;
    CHECK(cudaMalloc((void **) &d_clusters, N_SAMPLES * sizeof(int)));
    CHECK(cudaMemset(d_clusters, 0, N_SAMPLES * sizeof(int)));

    // # Dimensioni delle griglie.
    dim3 block_1d(min(BLOCK_SIZE, N_SAMPLES));
    dim3 grid_1d((N_SAMPLES + min(BLOCK_SIZE, N_SAMPLES) - 1) / BLOCK_SIZE);
    dim3 block_1d_k(min(BLOCK_SIZE, K));
    dim3 grid_1d_k((K + min(BLOCK_SIZE, K) - 1) / K);

    while (!stop_condition) {
        // # Inizializzo la mappa e la copio in GPU.
        init_map(h_map);
        h_map->size = N_SAMPLES;
        CHECK(cudaMemcpy(d_map, h_map, sizeof(struct KeyValueMap), cudaMemcpyHostToDevice));

        // # Assegno un cluster ad ogni nodo.
        d_assign_cluster<<<grid_1d, block_1d>>>(d_dataset, d_centers, d_map, d_clusters);
        CHECK(cudaDeviceSynchronize());
        CHECK(cudaMemcpy(h_clusters, d_clusters, N_SAMPLES * sizeof(int), cudaMemcpyDeviceToHost));

        // # Inizializzo i nuovi centri (con valore iniziale quello dei centri attuali) e gli copio in GPU.
        struct Centers *h_new_centers, *d_new_centers;
        h_new_centers = (struct Centers*) malloc(sizeof(struct Centers));
        copy_centers(h_new_centers, h_centers);
        CHECK(cudaMalloc((void **) &d_new_centers, sizeof(struct Centers)));
        CHECK(cudaMemcpy(d_new_centers, h_new_centers, sizeof(struct Centers), cudaMemcpyHostToDevice));

        // # Calcolo dei nuovi centroidi.
        d_update_centroids<<<grid_1d_k, block_1d_k>>>(d_map, d_new_centers);

        CHECK(cudaDeviceSynchronize());

        CHECK(cudaMemcpy(h_new_centers, d_new_centers, sizeof(struct Centers), cudaMemcpyDeviceToHost));

        // # Verifica della condizione di uscita (in termini di information gain inferiore alla soglia E).
        stop_condition = check_threshold(h_centers, h_new_centers);
        h_centers = h_new_centers;
        CHECK(cudaMemcpy(d_centers, d_new_centers, sizeof(struct Centers), cudaMemcpyDeviceToDevice));

        CHECK(cudaFree(d_new_centers));

        cont ++;
    }

    printf("  - Algoritmo terminato in %d iterazioni.\n  - Scrittura del dataset...\n", cont);

    // # Esecuzione dell'assegnamento un'ultima volta per associare un cluster ad ogni dato.
    init_map(h_map);
    CHECK(cudaMemcpy(d_map, h_map, sizeof(struct KeyValueMap), cudaMemcpyHostToDevice));

    d_assign_cluster<<<grid_1d, block_1d>>>(d_dataset, d_centers, d_map, d_clusters);

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(h_centers, d_centers, sizeof(struct Centers), cudaMemcpyDeviceToHost));

    // # Scrittura dei risultati.
    write_samples(h_dataset, h_clusters, true);

    printf("  - Scrittura completata.\n✔ PKMEANS GPU completato\n");

    free(h_map);
    free(h_centers);

    CHECK(cudaFree(d_map));
    CHECK(cudaFree(d_centers));
    CHECK(cudaFree(d_dataset));

    return make_timed_result(h_clusters);
}

// # Stampa i risultati ottenuti dalla CPU e dalla GPU e un confronto in termini di speedup.
int print_timed_results(TimedResult result_cpu, TimedResult result_gpu) {
    printf("Tempo totale CPU: %.2f\nTempo totale GPU: %.2f\n", result_cpu.total, result_gpu.total);
    printf("Speed-up: %.2f\n", result_cpu.total / result_gpu.total);

    return 0;
}

int main() {
    // # Allocazione del dataset.
    struct Samples *dataset;
    dataset = (struct Samples*) malloc(sizeof(struct Samples));

    // # Allocazione e inizializzazione dei centri.
    struct Centers *centers;
    centers = (struct Centers*) malloc(sizeof(struct Centers));
    init_centers(centers, dataset);

    // # Allocazione della struttura risultato: tempo di esecuzione + lista dei cluster.
    TimedResult result_cpu, result_gpu;
    double cpu_start, cpu_stop, gpu_start, gpu_stop;

    // # Esecuzione CPU.
    cpu_start = seconds();
    result_cpu = cpu_pkmeans(dataset, centers);
    cpu_stop = seconds() - cpu_start;
    result_cpu.total = cpu_stop;

    printf("\n");

    // # Esecuzione GPU.
    gpu_start = seconds();
    result_gpu = gpu_pkmeans(dataset, centers);
    gpu_stop = seconds() - gpu_start;
    result_gpu.total = gpu_stop;

    // # Controllo se i 2 array combaciano.
    printf("\nI risultati %s!\n", arrays_match(result_cpu.cluster, result_gpu.cluster) ? "COMBACIANO" : "NON COMBACIANO");

    // # Stampa delle informazioni utili (dimensione del blocco e tempi).
    printf("\n--- dimensione blocco = %d ---\n", BLOCK_SIZE);
    print_timed_results(result_cpu, result_gpu);

    return 0;
}

'File written in /content/src/pkmeans.cu'

In [13]:
!nvcc -arch=sm_75 -lineinfo src/pkmeans.cu -o pkmeans
!./pkmeans

PKMEANS CPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Inizio dell'algoritmo...
  - Algoritmo terminato in 94 iterazioni.
✔ PKMEANS CPU completato

PKMEANS GPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Inizio dell'algoritmo...
  - Algoritmo terminato in 94 iterazioni.
  - Scrittura del dataset...
  - Scrittura completata.
✔ PKMEANS GPU completato

I risultati COMBACIANO!

--- dimensione blocco = 1024 ---
Tempo totale CPU: 6.17
Tempo totale GPU: 1.27
Speed-up: 4.84


In [14]:
!nvprof ./pkmeans

PKMEANS CPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Inizio dell'algoritmo...
  - Algoritmo terminato in 94 iterazioni.
✔ PKMEANS CPU completato

PKMEANS GPU:
  - Definizione delle strutture dati...
==2699== NVPROF is profiling process 2699, command: ./pkmeans
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Inizio dell'algoritmo...
  - Algoritmo terminato in 94 iterazioni.
  - Scrittura del dataset...
  - Scrittura completata.
✔ PKMEANS GPU completato

I risultati COMBACIANO!

--- dimensione blocco = 1024 ---
Tempo totale CPU: 5.55
Tempo totale GPU: 1.12
Speed-up: 4.94
==2699== Profiling application: ./pkmeans
==2699== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   61.66%  336.68ms        94  3.5817ms  2.6090ms  8.8210ms  d_update_centroids(KeyValueMap*, Centers*)
     

In [15]:
!ncu -o pkmeans_profile pkmeans
!nsys profile -o pkmeans_report pkmeans

PKMEANS CPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Inizio dell'algoritmo...
  - Algoritmo terminato in 94 iterazioni.
✔ PKMEANS CPU completato

PKMEANS GPU:
  - Definizione delle strutture dati...
==PROF== Connected to process 2753 (/content/pkmeans)
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Inizio dell'algoritmo...
==PROF== Profiling "d_assign_cluster" - 0: 0%....50%....100% - 8 passes
==PROF== Profiling "d_update_centroids" - 1: 0%....50%....100% - 8 passes
==PROF== Profiling "d_assign_cluster" - 2: 0%....50%....100% - 8 passes
==PROF== Profiling "d_update_centroids" - 3: 0%....50%....100% - 8 passes
==PROF== Profiling "d_assign_cluster" - 4: 0%....50%....100% - 8 passes
==PROF== Profiling "d_update_centroids" - 5: 0%....50%....100% - 8 passes
==PROF== Profiling "d_assign_cluster" - 6: 0%....50%....100% - 8 passes
==PROF== Profili

# ✅ G-DBSCAN

In [16]:
%%cuda --name gdbscan.cu

#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/common.h"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph.h"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph.cpp"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph_d.cu"
#include "/content/drive/MyDrive/Istruzione/Università/Magistrale/Corsi/GPU computing/Progetto/spatial-staypoint-detection/lib/graph_d.h"
#include "/content/src/common.cu"
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define MIN_PTS 50
#define R 0.0925

#define BLOCK_SIZE 1024
#define BLOCK_SIZE_2D 32

// # Enumerativo per la classificazione dei punti per il clustering.
enum Type {
    Core, // # Punto focus di un nuovo cluster.
    Border, // # Nodo periferico.
    Noise // # Nodo sconnesso.
};

// # Struttura utile per la clusterizzazione dei nodi (esplorazione).
struct Nodes {
    int visited[N_SAMPLES];
    Type type[N_SAMPLES];
    int cluster[N_SAMPLES];
};

// # Struttura per tenere traccia dei vari tempi di esecuzione.
struct TimedResult {
    double graph_construction;
    double graph_bfs;
    double total;
    int* cluster;
};

// # Inizializza la struttura di esplorazione dei nodi.
int init_nodes(struct Nodes *nodes) {
    for (int i = 0; i < N_SAMPLES; i++) {
        nodes->visited[i] = 0;
        nodes->type[i] = Noise;
        nodes->cluster[i] = -1;
    }

    return 0;
}

// # Costruisce la struttura utile per tenere traccia dei tempi di esecuzione.
TimedResult make_timed_result(double construction_time, double bfs_time, int *cluster) {
    TimedResult result;

    result.graph_construction = construction_time;
    result.graph_bfs = bfs_time;
    result.cluster = cluster;

    return result;
}

// # Etichetta un nodo come core se questo ha collegati almeno min_pts nodi.
int classify_object(struct Nodes *nodes, node i, struct GraphStruct *graph, int min_pts) {
    if (graph->deg(i) > min_pts) {
        nodes->type[i] = Core;
    }

    return 0;
}

// # Costruisce il grafo dei cluster.
int make_graph(int min_pts, float r, struct Samples *dataset, struct GraphStruct *graph, struct Nodes *nodes) {
    // # Calcolo dei gradi.
    for (int i = 0; i < N_SAMPLES - 1; i++) {
        for (int j = i + 1; j < N_SAMPLES; j++) {
            if (calculate_distance(dataset->x[i], dataset->y[i], dataset->x[j], dataset->y[j]) <= r) {
                graph->cumDegs[i + 1]++;
                graph->cumDegs[j + 1]++;
                graph->edgeSize += 2;
            }
        }
    }

    // # Cumulazione reale dei gradi.
    for (int j = 0; j < N_SAMPLES; j++) {
        graph->cumDegs[j + 1] += graph->cumDegs[j];
    }

    // # Inizialzzazione e popolazione della lista dei vicini (lista unica per tutti i nodi, accesso con gradi cumulati).
    graph->neighs = (node *) malloc(sizeof(node) * graph->edgeSize);

    for (int i = 0; i < N_SAMPLES; i++) {
        int cont = 0;

        for (int j = 0; j < N_SAMPLES; j++) {
            if (i != j && calculate_distance(dataset->x[i], dataset->y[i], dataset->x[j], dataset->y[j]) <= r) {
                graph->neighs[graph->cumDegs[i] + cont] = j;

                cont++;
            }
        }
    }

    // # Rilevazione dei punti core.
    for (int i = 0; i < N_SAMPLES; i++) {
        classify_object(nodes, i, graph, min_pts);
    }

    return 0;
}

// # Controlla che l'array passato come parametro sia vuoto.
int array_is_empty(int *array) {
    for (int i = 0; i < N_SAMPLES; i++) {
        if (array[i] != 0) {
            return 0;
        }
    }

    return 1;
}

// # Esplora in ampiezza un singolo nodo.
int breadth_first_search_kernel(struct GraphStruct *graph, int *Fa, int *Xa, int idx) {
    if (Fa[idx]) {
        Fa[idx] = 0;
        Xa[idx] = 1;

        for (int i = 0; i < graph->deg(idx); i++) {
            int v = graph->neighs[graph->cumDegs[idx] + i];

            if (!Xa[v]) {
                Fa[v] = 1; // # Si inserisce il nodo nella frontiera di esplorazione.
            }
        }
    }

    return 0;
}

// # Si effettua la visita in ampiezza dell'intero grafo data una sorgente.
int breadth_first_search(int s, struct GraphStruct *graph, struct Nodes *nodes, int cluster) {
    int Xa[N_SAMPLES] = {0}; // # Maschera dei nodi esplorati (se Xa[v] == 1 significa che v è stato esplorato).
    int Fa[N_SAMPLES] = {0}; // # Maschera della frontiera di esplorazione.

    Fa[s] = 1;

    while (!array_is_empty(Fa)) {
        for (int i = 0; i < N_SAMPLES; i++) {
            breadth_first_search_kernel(graph, Fa, Xa, i); // # Esploro i nodi sulla frontiera.
        }
    }

    // # Si identificano i nodi periferici e si assegnano i cluster.
    for (int i = 0; i < N_SAMPLES; i++) {
        if (Xa[i] == 1) {
            nodes->cluster[i] = cluster;
            nodes->visited[i] = 1;

            if (nodes->type[i] != Core) {
                nodes->type[i] = Border;
            }
        }
    }

    return 0;
}

// # Si identificano i vari cluster (core dell'algoritmo).
int identify_cluster(struct GraphStruct *graph, struct Nodes *nodes) {
    int cluster = 0;

    // # Si visitano tutti i nodi se questi sono core.
    for (int i = 0; i < N_SAMPLES; i++) {
        if (nodes->visited[i] != 1 && nodes->type[i] == Core) {
            nodes->visited[i] = 1;
            nodes->cluster[i] = cluster; // # Ad ogni nodo core e collegati, si assegna un cluster.

            breadth_first_search(i, graph, nodes, cluster);

            cluster += 1;
        }
    }

    return 0;
}

// # Algoritmo g-dbscan su CPU.
TimedResult cpu_dbscan(struct Samples *dataset) {
    printf("DBSCAN CPU:\n  - Definizione delle strutture dati...\n");

    // # Inizializzo la struttura di esplorazione.
    struct Nodes *nodes;
    nodes = (struct Nodes*) malloc(sizeof(struct Nodes));

    // # Inizializzo il grafo di esplorazione.
    GraphStruct *graph = new GraphStruct();
    graph->cumDegs = (node_sz *) malloc(sizeof(node_sz) * (N_SAMPLES + 1));
    graph->nodeSize = N_SAMPLES;

    printf("  - Definizione completata.\n  - Inizializzazione in corso...\n");

    // # Lettura dei valori e inizializzazione delle strutture.
    read_samples(dataset);

    init_nodes(nodes);

    double construction_start, construction_stop; // # Inizializzazione dei tempi di costruzione del grafo.

    printf("  - Inizializzazione completata.\n  - Popolazione del grafo...\n");
    construction_start = seconds();
    make_graph(MIN_PTS, R, dataset, graph, nodes); // # Costruzione del grafo.
    construction_stop = seconds() - construction_start;

    double bfs_start, bfs_stop; // # Inizializzazione dei tempi di esplorazione del grafo.

    printf("  - Popolazione completata.\n  - Identificazione dei cluster...\n");
    bfs_start = seconds();
    identify_cluster(graph, nodes); // # Esplorazione del grafo.
    bfs_stop = seconds() - bfs_start;

    printf("  - Identificazione completata.\n✔ DBSCAN CPU completato\n");

    return make_timed_result(construction_stop, bfs_stop, nodes->cluster);
}

// # Kernel per il calcolo dei gradi di ogni nodo in GPU.
__global__ void calculate_degs(struct Samples *d_dataset, int *d_cum_degs, int *d_edge_size, float r) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;

    if (i >= N_SAMPLES || j >= N_SAMPLES) {
        return;
    }

    // # Due nodi sono vicini se sono a distanza <= r.
    if (i != j && d_calculate_distance(d_dataset->x[i], d_dataset->y[i], d_dataset->x[j], d_dataset->y[j]) <= r) {
        atomicAdd(&d_cum_degs[i], 1);
        atomicAdd(d_edge_size, 1);
    }
}

// # Kernel di scan naive per il calcolo dei gradi cumulati in GPU.
// # La riduzione avviene partendo da uno stride piccolo (elementi consecutiivi).
__global__ void naive_scan(int* input, int* output, int n) {
    extern __shared__ int temp[]; // # Pari a 2 volte la dimensione del blocco.

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int pout = 0, pin = 1;

    if (idx > N_SAMPLES) return;

    temp[pout * n + tid] = input[idx]; // # Caricamento dei dati relativi al blocco in shared.
    __syncthreads();

    for (int offset = 1; offset < n; offset *= 2) {
        pout = 1 - pout;
        pin = 1 - pout;

        if (tid >= offset)
            temp[pout * n + tid] = temp[pin * n + tid] + temp[pin * n + tid - offset]; // # Reduction di elementi distanti offset.
        else
            temp[pout * n + tid] = temp[pin * n + tid]; // # Copia del valore in posizione corretta per la prossima reduction.

        __syncthreads();
    }

    output[idx] = temp[pout * n + tid];
}

// # Kernel per la correzione dei gradi cumulati (dovuto al disallineamento per uso della shared memory) in GPU.
__global__ void add_block_sums(int* input, int* block_sums, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    if (idx < n && blockIdx.x > 0) {
        input[idx] += block_sums[blockIdx.x - 1];
    }
}

// # Kernel per l'aggiunta di uno zero all'inizio di un array in GPU.
__global__ void prepend_zero(int* source, int* destination) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    if (idx > N_SAMPLES) return;

    if (idx == 0) {
        destination[idx] = 0;
    } else {
        destination[idx] = source[idx - 1];
    }
}

// # Kernel per la creazione della lista di adiacenza generale (si ricorda l'accesso per grado) in GPU.
__global__ void make_adj_list(struct Samples *d_dataset, int *d_adj, int *d_cum_degs, float r) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i >= N_SAMPLES) return;

    int cont = 0;

    // # Si confronta il nodo sorgente (quello del kernel corrente) con tutti gli altri.
    for (int j = 0; j < N_SAMPLES; j++) {
        if (i != j && d_calculate_distance(d_dataset->x[i], d_dataset->y[i], d_dataset->x[j], d_dataset->y[j]) <= r) {
            d_adj[d_cum_degs[i] + cont] = j;
            cont++;
        }
    }
}

// # Kernel per la rilevazione dei nodi core (si ricorda la definizione di core).
__global__ void d_classify_objects(struct Nodes *d_nodes, int *d_cum_degs, int min_pts) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx >= N_SAMPLES) return;

    if (d_cum_degs[idx + 1] - d_cum_degs[idx] > min_pts) {
        d_nodes->type[idx] = Core;
    }
}


// # Costruisce il grafo sfruttando i vari kernel precedentemente definiti.
int make_graph_gpu(int min_pts, float r, struct Samples *h_dataset, struct GraphStruct *h_graph, struct Nodes *h_nodes) {
    // # Alloco su device il dataset.
    struct Samples *d_dataset;

    CHECK(cudaMalloc((void **) &d_dataset, sizeof(struct Samples)));
    CHECK(cudaMemcpy(d_dataset, h_dataset, sizeof(struct Samples), cudaMemcpyHostToDevice));

    // # Alloco il numero di archi.
    int h_edge_size = 0;
    int *d_edge_size;

    CHECK(cudaMalloc((void **) &d_edge_size, sizeof(int)));
    CHECK(cudaMemset(d_edge_size, 0, sizeof(int)));

    // # Alloco la lista dei gradi cumulati.
    int *h_cum_degs = (int*) malloc(sizeof(int) * (N_SAMPLES + 1));
    int *d_cum_degs;

    memset(h_cum_degs, 0, sizeof(int) * N_SAMPLES);

    CHECK(cudaMalloc((void **) &d_cum_degs, sizeof(int) * (N_SAMPLES + 1)));
    CHECK(cudaMemset(d_cum_degs, 0, sizeof(int) * (N_SAMPLES + 1)));

    // # Definisco le dimensioni di blocchi e griglie 2D.
    dim3 block_2d(min(N_SAMPLES, BLOCK_SIZE_2D), min(N_SAMPLES, BLOCK_SIZE_2D));
    dim3 grid_2d((N_SAMPLES + block_2d.x - 1) / block_2d.x, (N_SAMPLES + block_2d.y - 1) / block_2d.y);

    calculate_degs<<<grid_2d, block_2d>>>(d_dataset, d_cum_degs, d_edge_size, r); // # Calcolo dei gradi.

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(&h_edge_size, d_edge_size, sizeof(int), cudaMemcpyDeviceToHost));
    CHECK(cudaMemcpy(h_cum_degs, d_cum_degs, sizeof(int) * (N_SAMPLES + 1), cudaMemcpyDeviceToHost));

    // # Inizializzo la lista utile per il calcolo della lista di adiacenza globale.
    int* d_output;

    CHECK(cudaMalloc((void **) &d_output, sizeof(int) * (N_SAMPLES + 1)));
    CHECK(cudaMemset(d_output, 0, sizeof(int) * (N_SAMPLES + 1)));

    // # Definisco le dimensioni di blocchi e griglia 1D.
    dim3 block_1d(BLOCK_SIZE);
    dim3 grid_1d((N_SAMPLES + BLOCK_SIZE - 1) / BLOCK_SIZE);

    naive_scan<<<grid_1d, block_1d, BLOCK_SIZE * sizeof(int) * 2>>>(d_cum_degs, d_output, BLOCK_SIZE); // # Cumulazione dei gradi.

    CHECK(cudaDeviceSynchronize());

    // # Inizializzo le strutture per la correzione dei gradi cumulati.
    int n_blocks = grid_1d.x;
    int *h_block_sums = (int *) malloc(sizeof(int) * n_blocks);

    for (int i = 0; i < n_blocks; i++) {
        int index = i * BLOCK_SIZE + BLOCK_SIZE - 1;

        if (index < N_SAMPLES + 1) {
            CHECK(cudaMemcpy(&h_block_sums[i], &d_output[index], sizeof(int), cudaMemcpyDeviceToHost)); // # Si memorizzano i valori degli ultimi elementi di ogni blocco.
        }
    }

    // # Cumulo gli ultimi elementi di ogni blocco.
    for (int i = 1; i < n_blocks; i++) {
        h_block_sums[i] += h_block_sums[i - 1];
    }

    // # Inizializzo la struttura degli ultimi elementi dei blocchi cumulata in GPU.
    int* d_block_sums;

    CHECK(cudaMalloc((void **) &d_block_sums, n_blocks * sizeof(int)));
    CHECK(cudaMemcpy(d_block_sums, h_block_sums, n_blocks * sizeof(int), cudaMemcpyHostToDevice));

    add_block_sums<<<grid_1d, block_1d>>>(d_output, d_block_sums, N_SAMPLES); // # Applicazione parallela della correzione.

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(h_cum_degs, d_output, sizeof(int) * (N_SAMPLES + 1), cudaMemcpyDeviceToHost));

    // # Inizializzo la variabile finale contenente i gradi cumulati.
    int *d_final_cum_degs;

    CHECK(cudaMalloc((void **) &d_final_cum_degs, (N_SAMPLES + 1) * sizeof(int)));
    CHECK(cudaMemset(d_final_cum_degs, 0, sizeof(int) * (N_SAMPLES + 1)));

    prepend_zero<<<grid_1d, block_1d>>>(d_output, d_final_cum_degs); // # Aggiunta di uno zero in testa.

    CHECK(cudaMemcpy(h_cum_degs, d_final_cum_degs, sizeof(int) * (N_SAMPLES + 1), cudaMemcpyDeviceToHost));

    h_graph->cumDegs = (node_sz *) h_cum_degs;
    h_graph->edgeSize = h_edge_size;

    // # Inizializzo le strutture per la creazione della lista di adiacenza.
    int *h_adj = (int*) malloc(sizeof(int) * h_graph->edgeSize);
    int *d_adj;

    CHECK(cudaMalloc((void **) &d_adj, sizeof(int) * h_graph->edgeSize));
    CHECK(cudaMemset(d_adj, 0, sizeof(int) * h_graph->edgeSize));

    make_adj_list<<<grid_1d, block_1d>>>(d_dataset, d_adj, d_final_cum_degs, r); // # Costruzione della lista di adiacenza.

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(h_adj, d_adj, sizeof(int) * h_graph->edgeSize, cudaMemcpyDeviceToHost));

    h_graph->neighs = (node *) h_adj;

    // # Iizializzo la struttura utile per l'esplorazione del grafo costruito.
    struct Nodes *d_nodes;

    CHECK(cudaMalloc((void **) &d_nodes, sizeof(struct Nodes)));
    CHECK(cudaMemcpy(d_nodes, h_nodes, sizeof(struct Nodes), cudaMemcpyHostToDevice));

    d_classify_objects<<<grid_1d, block_1d>>>(d_nodes, d_final_cum_degs, min_pts); // # Classificazione dei nodi core.

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(h_nodes, d_nodes, sizeof(struct Nodes), cudaMemcpyDeviceToHost));

    CHECK(cudaFree(d_dataset));
    CHECK(cudaFree(d_edge_size));
    CHECK(cudaFree(d_cum_degs));
    CHECK(cudaFree(d_output));
    CHECK(cudaFree(d_final_cum_degs));
    CHECK(cudaFree(d_adj));
    CHECK(cudaFree(d_nodes));

    return 0;
}

// # Kernel per l'esplorazione del sotto-grafo relativo ai nodi.
__global__ void breadth_first_search_gpu_kernel(node *d_neighs, node_sz *d_cum_degs, int *Fa, int *Xa) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    if (idx >= N_SAMPLES) return;

    if (Fa[idx]) {
        Fa[idx] = 0;
        Xa[idx] = 1;

        for (int i = 0; i < d_cum_degs[idx + 1] - d_cum_degs[idx]; i++) {
            int v = d_neighs[d_cum_degs[idx] + i];

            if (!Xa[v]) {
                Fa[v] = 1; // # Si inserisce il nodo nella frontiera di esplorazione.
            }
        }
    }
}

// # Kernel per l'assegnamento dei cluster ai nodi parte del sotto-grafo di uno core.
__global__ void assign_cluster(int * d_Xa, struct Nodes * d_nodes, int cluster) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    if (idx >= N_SAMPLES) return;

    if (d_Xa[idx] == 1) {
        d_nodes->cluster[idx] = cluster;
        d_nodes->visited[idx] = 1;

        if (d_nodes->type[idx] != Core) {
            d_nodes->type[idx] = Border;
        }
    }
}

// # Effettua l'esplorazione in ampiezza di un grafo sfruttando i kernel definiti in precedenza.
int breadth_first_search_gpu(int s, struct GraphStruct *h_graph, struct Nodes *h_nodes, int cluster) {
    int h_Xa[N_SAMPLES] = {0}; // # Maschera dei nodi esplorati (se Xa[v] == 1 significa che v è stato esplorato).
    int h_Fa[N_SAMPLES] = {0}; // # Maschera della frontiera di esplorazione.

    h_Fa[s] = 1;

    // # Alloco su device le due maschere.
    int *d_Fa, *d_Xa;

    CHECK(cudaMalloc((void **) &d_Fa, sizeof(int) * N_SAMPLES));
    CHECK(cudaMemcpy(d_Fa, &h_Fa, sizeof(int) * N_SAMPLES, cudaMemcpyHostToDevice));

    CHECK(cudaMalloc((void **) &d_Xa, sizeof(int) * N_SAMPLES));
    CHECK(cudaMemcpy(d_Xa, &h_Xa, sizeof(int) * N_SAMPLES, cudaMemcpyHostToDevice));

    // # Alloco su device la lista di adiacenza globale.
    node *d_neighs;

    CHECK(cudaMalloc((void **) &d_neighs, sizeof(int) * h_graph->edgeSize));
    CHECK(cudaMemcpy(d_neighs, h_graph->neighs, sizeof(int) * h_graph->edgeSize, cudaMemcpyHostToDevice));

    // # Alloco su device la lista dei gradi cumulati.
    node_sz *d_cum_degs;

    CHECK(cudaMalloc((void **) &d_cum_degs, sizeof(int) * (N_SAMPLES + 1)));
    CHECK(cudaMemcpy(d_cum_degs, h_graph->cumDegs, sizeof(int) * (N_SAMPLES + 1), cudaMemcpyHostToDevice));

    // # Alloco su device la struttura utile all'esplorazione del grafo.
    struct Nodes *d_nodes;

    CHECK(cudaMalloc((void **) &d_nodes, sizeof(struct Nodes)));
    CHECK(cudaMemcpy(d_nodes, h_nodes, sizeof(struct Nodes), cudaMemcpyHostToDevice));

    // # Definisco le dimensioni di blocco e griglia.
    dim3 block(min(N_SAMPLES, BLOCK_SIZE));
    dim3 grid((N_SAMPLES + block.x - 1) / block.x);

    while (!array_is_empty(h_Fa)) {
        breadth_first_search_gpu_kernel<<<grid, block>>>(d_neighs, d_cum_degs, d_Fa, d_Xa); // # Effettuo l'esplorazione dei nodi sulla frontiera.

        CHECK(cudaDeviceSynchronize());

        CHECK(cudaMemcpy(h_Fa, d_Fa, sizeof(int) * N_SAMPLES, cudaMemcpyDeviceToHost)); // # La frontiera deve essere sempre aggiornata sull'host.
    }

    CHECK(cudaDeviceSynchronize());

    // # Si effettua l'assegnazione dei cluster.
    assign_cluster<<<grid, block>>>(d_Xa, d_nodes, cluster);

    CHECK(cudaMemcpy(h_nodes, d_nodes, sizeof(struct Nodes), cudaMemcpyDeviceToHost));

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaFree(d_Fa));
    CHECK(cudaFree(d_Xa));
    CHECK(cudaFree(d_nodes));

    return 0;
}

// # Assegna i cluster ai differenti sotto-grafi.
int identify_cluster_gpu(struct GraphStruct *h_graph, struct Nodes *h_nodes) {
    int cluster = 0;

    for (int i = 0; i < N_SAMPLES; i++) {
        if (h_nodes->visited[i] != 1 && h_nodes->type[i] == Core) {
            h_nodes->visited[i] = 1;
            h_nodes->cluster[i] = cluster; // # Ad ogni nodo core e collegati, si assegna un cluster.

            breadth_first_search_gpu(i, h_graph, h_nodes, cluster);

            cluster += 1;
        }
    }

    return 0;
}

// # Algoritmo g-dbscan su GPU.
TimedResult gpu_dbscan(struct Samples *dataset) {
    printf("DBSCAN GPU:\n  - Definizione delle strutture dati...\n");

    // # Inizializzazione della struttura utile per l'esplorazione.
    struct Nodes *nodes;
    nodes = (struct Nodes*) malloc(sizeof(struct Nodes));

    // # Inizializzazione del grafo.
    GraphStruct *graph = new GraphStruct();
    graph->cumDegs = (node_sz *) malloc(sizeof(node_sz) * (N_SAMPLES + 1));
    graph->nodeSize = N_SAMPLES;

    printf("  - Definizione completata.\n  - Inizializzazione in corso...\n");

    // # Lettura dei campioni ed inizializzazione.
    read_samples(dataset);

    init_nodes(nodes);

    double construction_start, construction_stop; // # Inizializzazione dei tempi di costruzione del grafo.

    printf("  - Inizializzazione completata.\n  - Popolazione del grafo...\n");
    construction_start = seconds();
    make_graph_gpu(MIN_PTS, R, dataset, graph, nodes); // # Costruzione del grafo.
    construction_stop = seconds() - construction_start;

    double bfs_start, bfs_stop; // # Inizializzazione dei tempi di esplorazione del grafo.

    printf("  - Popolazione completata.\n  - Identificazione dei cluster...\n");
    bfs_start = seconds();
    identify_cluster_gpu(graph, nodes); // # Esplorazione del grafo.
    bfs_stop = seconds() - bfs_start;

    printf("  - Identificazione completata.\n  - Scrittura del dataset...\n");
    write_samples(dataset, nodes->cluster, false); // # Salvataggio dei risultati.

    printf("  - Scrittura completata.\n✔ DBSCAN GPU completato\n");

    return make_timed_result(construction_stop, bfs_stop, nodes->cluster);
}

// # Stampa i differenti tempi di esecuzione (nel dettaglio e totali) oltre che allo speedup ottenuto.
int print_timed_results(TimedResult result_cpu, TimedResult result_gpu) {
    printf("Tempo di costruzione del grafo CPU: %.2f\nTempo di costruzione del grafo GPU: %.2f\n", result_cpu.graph_construction, result_gpu.graph_construction);
    printf("Speed-up: %.2f\n", result_cpu.graph_construction / result_gpu.graph_construction);

    printf("\nTempo di esplorazione del grafo CPU: %.2f\nTempo di esplorazione del grafo GPU: %.2f\n", result_cpu.graph_bfs, result_gpu.graph_bfs);
    printf("Speed-up: %.2f\n", result_cpu.graph_bfs / result_gpu.graph_bfs);

    printf("\nTempo totale CPU: %.2f\nTempo totale GPU: %.2f\n", result_cpu.total, result_gpu.total);
    printf("Speed-up: %.2f\n", result_cpu.total / result_gpu.total);

    return 0;
}

int main() {
    // # Inizializzo la struttura contenente i sample.
    struct Samples *dataset;
    dataset = (struct Samples*) malloc(sizeof(struct Samples));

    // # Inizializzo le variabili contententi i tempi.
    TimedResult result_cpu, result_gpu;
    double cpu_start, cpu_stop, gpu_start, gpu_stop;

    // # Esecuzione su CPU.
    cpu_start = seconds();
    result_cpu = cpu_dbscan(dataset);
    cpu_stop = seconds() - cpu_start;

    result_cpu.total = cpu_stop;

    printf("\n");

    // # Esecuzione su GPU.
    gpu_start = seconds();
    result_gpu = gpu_dbscan(dataset);
    gpu_stop = seconds() - gpu_start;

    result_gpu.total = gpu_stop;

    // # Controllo se i 2 array combaciano.
    printf("\nI risultati %s!\n", arrays_match(result_cpu.cluster, result_gpu.cluster) ? "COMBACIANO" : "NON COMBACIANO");

    // # Stampa delle informazioni utili (dimensione del blocco e tempi).
    printf("\n--- dimensione blocco 1D = %d, dimensione blocco 2D = %d ---\n", BLOCK_SIZE, BLOCK_SIZE_2D);
    print_timed_results(result_cpu, result_gpu);

    return 0;
}

'File written in /content/src/gdbscan.cu'

In [17]:
!nvcc -arch=sm_75 -lineinfo src/gdbscan.cu -o gdbscan
!./gdbscan

DBSCAN CPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Popolazione del grafo...
  - Popolazione completata.
  - Identificazione dei cluster...
  - Identificazione completata.
✔ DBSCAN CPU completato

DBSCAN GPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Popolazione del grafo...
  - Popolazione completata.
  - Identificazione dei cluster...
  - Identificazione completata.
  - Scrittura del dataset...
  - Scrittura completata.
✔ DBSCAN GPU completato

I risultati COMBACIANO!

--- dimensione blocco 1D = 1024, dimensione blocco 2D = 32 ---
Tempo di costruzione del grafo CPU: 117.71
Tempo di costruzione del grafo GPU: 4.18
Speed-up: 28.13

Tempo di esplorazione del grafo CPU: 0.04
Tempo di esplorazione del grafo GPU: 0.12
Speed-up: 0.38

Tempo totale CPU: 117.79
Tempo totale GPU: 4.76
Speed-up: 24.73


In [18]:
!nvprof ./gdbscan

DBSCAN CPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Popolazione del grafo...
  - Popolazione completata.
  - Identificazione dei cluster...
  - Identificazione completata.
✔ DBSCAN CPU completato

DBSCAN GPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Popolazione del grafo...
==3612== NVPROF is profiling process 3612, command: ./gdbscan
  - Popolazione completata.
  - Identificazione dei cluster...
  - Identificazione completata.
  - Scrittura del dataset...
  - Scrittura completata.
✔ DBSCAN GPU completato

I risultati COMBACIANO!

--- dimensione blocco 1D = 1024, dimensione blocco 2D = 32 ---
Tempo di costruzione del grafo CPU: 116.52
Tempo di costruzione del grafo GPU: 4.37
Speed-up: 26.66

Tempo di esplorazione del grafo CPU: 0.04
Tempo di esplorazione del grafo GPU: 0.13
Speed-up: 0.32

Temp

In [19]:
!ncu -o gdbscan_profile gdbscan
!nsys profile -o gdbscan_report gdbscan

DBSCAN CPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Popolazione del grafo...
  - Popolazione completata.
  - Identificazione dei cluster...
  - Identificazione completata.
✔ DBSCAN CPU completato

DBSCAN GPU:
  - Definizione delle strutture dati...
  - Definizione completata.
  - Inizializzazione in corso...
  - Inizializzazione completata.
  - Popolazione del grafo...
==PROF== Connected to process 4136 (/content/gdbscan)
==PROF== Profiling "calculate_degs" - 0: 0%....50%....100% - 8 passes
==PROF== Profiling "naive_scan(int *, int *, int)" - 1: 0%....50%....100% - 8 passes
==PROF== Profiling "add_block_sums" - 2: 0%....50%....100% - 8 passes
==PROF== Profiling "prepend_zero(int *, int *)" - 3: 0%....50%....100% - 8 passes
==PROF== Profiling "make_adj_list" - 4: 0%....50%....100% - 8 passes
==PROF== Profiling "d_classify_objects" - 5: 0%....50%....100% - 8 passes
  - Popolazione completata